# H2O AutoML 라이브 데모

> RAPIDS LAB 세미나 — Pandas 이후, RAPIDS 이전의 자동화 병렬 ML 프레임워크

---

## 발표 흐름
1. **[Phase 1]** H2O 초기화 + 데이터 로드 + AutoML 학습 시작 (~1분)
2. **[Phase 2]** 학습 대기 중 → README / 슬라이드로 H2O AutoML 설명 (~5분)
3. **[Phase 3]** 학습 완료 후 → 결과 분석 (~4분)

---
## Phase 1: 런타임 시작
### 1-1. H2O 클러스터 초기화

In [ ]:
import h2o
from h2o.automl import H2OAutoML

# H2O 클러스터 시작 (로컬 JVM)
# nthreads=-1: 모든 CPU 코어 사용 (멀티코어 병렬 처리의 핵심)
# max_mem_size: JVM 힙 메모리 (XGBoost는 별도 메모리 필요하므로 전체의 2/3 권장)
h2o.init(nthreads=-1, max_mem_size="4G")

### 1-2. 데이터셋 로드

**Airlines Dataset** (~580MB, 전체 데이터의 5% 샘플)
- 1987~2008년 미국 항공편 데이터
- 타겟: `IsDepDelayed` (출발 지연 여부, YES/NO)
- 약 500만 건 레코드
- 원본: `s3://h2o-public-test-data/bigdata/laptop/airlines_all.05p.csv`

In [ ]:
# H2O 공식 제공 Airlines 데이터셋 (~580MB, 전체의 5%)
# 로컬: datasets/airlines_all.05p.csv
# 원본: https://s3.amazonaws.com/h2o-public-test-data/bigdata/laptop/airlines_all.05p.csv
airlines = h2o.import_file("../datasets/airlines_all.05p.csv")

print(f"데이터 크기: {airlines.shape}")
print(f"컬럼: {airlines.columns}")
airlines.head(5)

In [ ]:
# 데이터 기본 탐색
airlines.describe()

### 1-3. AutoML 학습 시작

**핵심 포인트**: 이 셀 하나로 6가지 알고리즘 + 하이퍼파라미터 튜닝 + 앙상블이 전부 자동 수행됩니다.

In [ ]:
# 예측 변수 / 응답 변수 설정
y = "IsDepDelayed"
x = airlines.columns
x.remove(y)

# 응답 변수를 factor로 변환 (이진분류)
airlines[y] = airlines[y].asfactor()

# Train/Test 분리 (80/20)
train, test = airlines.split_frame(ratios=[0.8], seed=42)
print(f"Train: {train.shape}, Test: {test.shape}")

In [ ]:
%%time

# ============================================================
# H2O AutoML 실행 — 이것이 전부입니다
# ============================================================
# max_runtime_secs=300 : 5분 동안 가능한 많은 모델 탐색
# seed=42             : 재현성 (단, DeepLearning은 비결정적)
# nfolds=5            : 5-fold cross-validation (기본값)
#
# 내부에서 수행되는 작업:
#   1) XGBoost 사전 정의 모델 3개
#   2) GLM (lambda_search 포함)
#   3) Random Forest (DRF)
#   4) GBM 5개
#   5) Deep Neural Network
#   6) Extremely Randomized Trees (XRT)
#   7) XGBoost/GBM/DNN 랜덤 그리드 서치
#   8) Stacked Ensemble 2종 (Best of Family + All Models)
# ============================================================

aml = H2OAutoML(
    max_runtime_secs=300,
    seed=42,
    project_name="airlines_automl",
    sort_metric="AUC",
    verbosity="info",
)

aml.train(x=x, y=y, training_frame=train)

print("\n=== AutoML 학습 완료 ===")

---

> **여기서 학습이 돌아가는 동안 README.md로 이동하여 H2O AutoML 특징을 설명합니다.**
> 
> (Phase 2: 약 5분 설명)

---

## Phase 3: 결과 분석

### 3-1. 리더보드 확인

AutoML이 학습한 모든 모델의 성능 순위표입니다.

In [ ]:
# 리더보드 — 모든 모델의 CV 성능 비교
lb = aml.leaderboard
print(f"총 {lb.nrows}개 모델 학습 완료\n")
lb.head(rows=lb.nrows)

In [ ]:
# 확장 리더보드: 학습 시간 + 예측 속도 포함
lb_ext = h2o.automl.get_leaderboard(aml, extra_columns="ALL")
lb_ext.head(rows=lb_ext.nrows)

### 3-2. 최고 모델 (Leader) 분석

In [ ]:
# AutoML이 선택한 최고 모델
leader = aml.leader
print(f"Best Model: {leader.model_id}")
print(f"Algorithm:  {leader.algo}")
print(f"\n--- Model Parameters ---")
leader.params

In [ ]:
# 테스트 데이터에서 성능 평가
perf = leader.model_performance(test)
print(perf)

In [ ]:
# AUC 확인
print(f"Test AUC: {perf.auc():.4f}")
print(f"Test Logloss: {perf.logloss():.4f}")

### 3-3. 알고리즘별 최고 모델 비교

In [ ]:
# 알고리즘별 최고 모델 가져오기
algos = ["xgboost", "gbm", "glm", "deeplearning", "drf", "stackedensemble"]

print(f"{'Algorithm':<20} {'Model ID':<55} {'AUC':>8}")
print("-" * 85)

for algo in algos:
    try:
        m = aml.get_best_model(algorithm=algo)
        p = m.model_performance(test)
        print(f"{algo:<20} {m.model_id:<55} {p.auc():>8.4f}")
    except Exception as e:
        print(f"{algo:<20} {'N/A':<55} {str(e)[:8]:>8}")

### 3-4. Feature Importance

In [ ]:
# 리더 모델의 변수 중요도 (tree 기반 모델인 경우)
# StackedEnsemble인 경우 base learner 중 하나를 사용
best_non_ensemble = aml.get_best_model(algorithm="gbm")
if best_non_ensemble is not None:
    best_non_ensemble.varimp_plot()

### 3-5. 예측 수행

In [ ]:
# 테스트 데이터 예측
preds = aml.predict(test)
preds.head(10)

### 3-6. AutoML 이벤트 로그

학습 과정에서 어떤 일이 일어났는지 확인합니다.

In [ ]:
# AutoML 이벤트 로그
aml.event_log.head(rows=30)

In [ ]:
# 학습 정보 요약
print("=== Training Info ===")
for k, v in aml.training_info.items():
    print(f"  {k}: {v}")

---

## 정리

| 항목 | 결과 |
|------|------|
| 코드 라인 수 | AutoML 실행: **단 6줄** |
| 탐색 알고리즘 | GBM, XGBoost, DRF, GLM, DeepLearning, StackedEnsemble |
| 자동 수행 항목 | 하이퍼파라미터 튜닝, Cross-validation, Early Stopping, 앙상블 |
| 병렬 처리 | JVM 기반 멀티코어 자동 활용 (설정 불필요) |

> **"5분, 6줄의 코드로 수십 개의 모델을 자동 학습하고 최적 앙상블까지 구성"**
> — 이것이 RAPIDS 이전 시대, H2O AutoML이 제시한 답입니다.

In [ ]:
# 클러스터 종료
# h2o.cluster().shutdown()